# Week 6: Regression Models using `Iris.csv`

This notebook demonstrates:

1. Data upload and inspection
2. Simple Linear Regression
3. Multiple Linear Regression
4. Binary Logistic Regression
5. Prediction
6. Confusion Matrix
7. Basic visualization

**Important:** The uploaded `Iris.csv` has these columns:

- `Id`
- `SepalLengthCm`
- `SepalWidthCm`
- `PetalLengthCm`
- `PetalWidthCm`
- `Species`

The original dataset contains 3 species. For Logistic Regression, we will use **Iris-setosa vs Iris-versicolor** as a binary classification problem.


## 1. Install and load required packages

Run this cell once if the packages are not already installed.

In [3]:
# Install packages if needed
packages <- c("tidyverse", "caret")

installed <- rownames(installed.packages())

for (p in packages) {
  if (!(p %in% installed)) {
    install.packages(p, repos = "https://cloud.r-project.org")
  }
}

library(tidyverse)
library(caret)


Installing package into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘listenv’, ‘parallelly’, ‘future’, ‘globals’, ‘shape’, ‘future.apply’, ‘numDeriv’, ‘progressr’, ‘SQUAREM’, ‘diagram’, ‘lava’, ‘prodlim’, ‘proxy’, ‘iterators’, ‘clock’, ‘gower’, ‘hardhat’, ‘ipred’, ‘sparsevctrs’, ‘timeDate’, ‘e1071’, ‘foreach’, ‘ModelMetrics’, ‘plyr’, ‘pROC’, ‘recipes’, ‘reshape2’


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.2.1     ✔ readr     2.2.0
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.3     ✔ tibble    3.3.1
✔ lubridate 1.9.5     ✔ tidyr     1.3.2
✔ purrr     1.2.2     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: lattice


Attaching package: ‘care

## 2. Upload and read `Iris.csv`

In Google Colab, upload the provided `Iris.csv` file to `/content/`.

If the file is already uploaded, just run the next cell.


In [4]:
# Read the uploaded CSV file
iris_data <- read.csv("/content/Iris.csv")

# Display first 6 rows
head(iris_data)


Warning message in file(file, "rt"):
“cannot open file '/content/Iris.csv': No such file or directory”


ERROR: Error in file(file, "rt"): cannot open the connection


In [ ]:
# Check the structure of the dataset
str(iris_data)

# Dataset dimensions
dim(iris_data)

# Summary statistics
summary(iris_data)


# A. Simple Linear Regression

We will predict **PetalLengthCm** using **SepalLengthCm**.

- Response / Dependent variable: `PetalLengthCm`
- Predictor / Independent variable: `SepalLengthCm`

Because `PetalLengthCm` is continuous, Linear Regression is appropriate.


In [ ]:
model_slr <- lm(
  PetalLengthCm ~ SepalLengthCm,
  data = iris_data
)

summary(model_slr)


### Visualize Simple Linear Regression

In [ ]:
ggplot(iris_data, aes(x = SepalLengthCm, y = PetalLengthCm)) +
  geom_point(aes(color = Species)) +
  geom_smooth(method = "lm", se = TRUE) +
  theme_minimal() +
  labs(
    title = "Simple Linear Regression",
    x = "Sepal Length (cm)",
    y = "Petal Length (cm)"
  )


# B. Multiple Linear Regression

Now we will predict **PetalLengthCm** using three predictors:

- `SepalLengthCm`
- `SepalWidthCm`
- `PetalWidthCm`

This is Multiple Linear Regression because there is more than one predictor.


In [ ]:
model_mlr <- lm(
  PetalLengthCm ~ SepalLengthCm + SepalWidthCm + PetalWidthCm,
  data = iris_data
)

summary(model_mlr)


### Predict Petal Length for a New Flower

In [ ]:
new_data <- data.frame(
  SepalLengthCm = 5.5,
  SepalWidthCm = 3.0,
  PetalWidthCm = 1.2
)

predicted_petal_length <- predict(model_mlr, newdata = new_data)

predicted_petal_length


# C. Binary Logistic Regression

The original Iris dataset has **3 classes**:

- Iris-setosa
- Iris-versicolor
- Iris-virginica

Standard Logistic Regression here will be demonstrated as a **binary classification** problem.

We will keep:

- `Iris-setosa` → 0
- `Iris-versicolor` → 1

and remove `Iris-virginica`.


In [ ]:
iris_binary <- iris_data %>%
  filter(Species %in% c("Iris-setosa", "Iris-versicolor")) %>%
  mutate(
    is_versicolor = ifelse(Species == "Iris-versicolor", 1, 0)
  )

head(iris_binary)

table(iris_binary$Species)


### Build the Logistic Regression Model

We will predict whether the flower is **Iris-versicolor** using the four measurement variables.


In [ ]:
model_logit <- glm(
  is_versicolor ~ SepalLengthCm + SepalWidthCm + PetalLengthCm + PetalWidthCm,
  data = iris_binary,
  family = binomial()
)

summary(model_logit)


### Predicted Probabilities

`type = "response"` returns the probability of belonging to the class `Iris-versicolor`.


In [ ]:
probs <- predict(
  model_logit,
  type = "response"
)

head(probs)


### Convert Probabilities into Classes

We use **0.5 as the classification threshold**:

- Probability > 0.5 → Iris-versicolor (1)
- Probability ≤ 0.5 → Iris-setosa (0)


In [ ]:
preds <- ifelse(probs > 0.5, 1, 0)

table(
  Predicted = preds,
  Actual = iris_binary$is_versicolor
)


### Confusion Matrix

In [ ]:
confusionMatrix(
  factor(preds, levels = c(0, 1)),
  factor(iris_binary$is_versicolor, levels = c(0, 1)),
  positive = "1"
)


# D. Predict a New Flower

Let's predict a new flower using the Logistic Regression model.

The measurements below are the same as the example used for Linear Regression.


In [ ]:
new_flower <- data.frame(
  SepalLengthCm = 5.5,
  SepalWidthCm = 3.0,
  PetalLengthCm = 4.0,
  PetalWidthCm = 1.2
)

new_probability <- predict(
  model_logit,
  newdata = new_flower,
  type = "response"
)

new_class <- ifelse(
  new_probability > 0.5,
  "Iris-versicolor",
  "Iris-setosa"
)

new_probability
new_class


# Summary

| Model | Target Variable | Output |
|---|---|---|
| Simple Linear Regression | `PetalLengthCm` | Continuous value |
| Multiple Linear Regression | `PetalLengthCm` | Continuous value |
| Binary Logistic Regression | `Species` | Class / Probability |

### Key idea

The **same dataset** can be used for different models depending on the target variable and the question we want to answer.

- Linear Regression → **How much? / What numerical value?**
- Logistic Regression → **Which class? / What is the probability?**
